# 🔵 DATA NAPOLI — Orchestratore Definitivo Multi-Fonte
Questo notebook combina l'architettura aziendale (Google Drive, SQLite, Riconciliazione) con gli estrattori di precisione per 10maggio87, StadiaPostcards e FBref.
Esegui le celle in sequenza per estrarre la storia del Napoli.

In [ ]:
# @title 1. Installazione dipendenze
!pip install requests beautifulsoup4 lxml pandas soccerdata nest-asyncio rapidfuzz
print("✅ Dipendenze installate.")

In [ ]:
# @title 2. Montaggio Google Drive e Setup Cartelle
import os
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/DataNapoli"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/raw", exist_ok=True)       # HTML grezzi/JSON
os.makedirs(f"{BASE_DIR}/processed", exist_ok=True) # JSON puliti per frontend
os.makedirs(f"{BASE_DIR}/db", exist_ok=True)        # Database SQLite
print(f"✅ Drive montato. Cartelle create in {BASE_DIR}")

In [ ]:
# @title 3. Inizializzazione Database SQLite
import sqlite3
import json
import os

DB_PATH = f"{BASE_DIR}/db/data_napoli.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS matches (
            match_id TEXT PRIMARY KEY,
            date TEXT,
            season TEXT,
            competizione TEXT,
            squadra_casa TEXT,
            squadra_trasferta TEXT,
            home_goals INTEGER,
            away_goals INTEGER,
            stadio TEXT,
            affluenza INTEGER,
            scorers TEXT,
            fonti TEXT,
            note TEXT
        )
    """)
    conn.commit()
    conn.close()
    print(f"✅ Database SQLite pronto: {DB_PATH}")

init_db()

In [ ]:
# @title 4. Estrattore 10maggio87 (Backbone Storico tramite __NEXT_DATA__)
import requests
import json
import re
import os
from bs4 import BeautifulSoup

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

def parse_date(date_str):
    try:
        # Converte da DD/MM/YYYY a YYYY-MM-DD
        parts = str(date_str).split('/')
        if len(parts) == 3:
            return f"{parts[2]}-{parts[1].zfill(2)}-{parts[0].zfill(2)}"
        return str(date_str)
    except:
        return str(date_str)

def scrape_10maggio87(season):
    url = f'https://www.10maggio87.it/Almanacco/Napoli/{season}/'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200: return []
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        script_tag = soup.find('script', id='__NEXT_DATA__')
        if not script_tag: return []
        
        next_data = json.loads(script_tag.string)
        page_props = next_data.get('props', {}).get('pageProps', {})
        
        # Salviamo il raw data sul drive (Zero Loss rule)
        raw_path = f"{BASE_DIR}/raw/10maggio87_{season}.json"
        with open(raw_path, 'w', encoding='utf-8') as f:
            json.dump(page_props, f, ensure_ascii=False)

        matches = []
        
        # Il nuovo estrattore cerca le partite in qualsiasi "card" della pagina
        def find_matches_in_node(obj):
            found = []
            if isinstance(obj, dict):
                for k, v in obj.items():
                    found.extend(find_matches_in_node(v))
            elif isinstance(obj, list):
                if len(obj) > 0 and isinstance(obj[0], dict):
                    keys = set(obj[0].keys())
                    # Le partite su 10maggio87 hanno queste chiavi esatte:
                    if keys & {'competition', 'homeTeam', 'result', 'dateLong', 'teams'}:
                        found.extend(obj)
                for item in obj:
                    found.extend(find_matches_in_node(item))
            return found

        raw_matches = find_matches_in_node(page_props)
        
        # Filtriamo eventuali duplicati (stessa data e stesse squadre)
        seen = set()
        for raw_m in raw_matches:
            # Evitiamo elementi che per caso hanno chiavi simili ma non sono partite
            if 'result' not in raw_m or 'competition' not in raw_m: continue
            
            teams_str = raw_m.get('teams', '')
            if ' - ' in teams_str:
                home_team, away_team = teams_str.split(' - ', 1)
            else:
                home_team = raw_m.get('homeTeam', '')
                away_team = "Napoli" if home_team != "Napoli" else "Avversario"

            date_clean = parse_date(raw_m.get('dateLong', '1926-01-01'))
            uid = f"{date_clean}-{home_team}"
            if uid in seen: continue
            seen.add(uid)

            m = {
                'source': '10maggio87',
                'season': season,
                'competizione': raw_m.get('competition', 'ND'),
                'date': date_clean,
                'squadra_casa': home_team.strip(),
                'squadra_trasferta': away_team.strip(),
            }
            
            res = str(raw_m.get('result', ''))
            if '-' in res:
                parts = res.split('-')
                m['home_goals'] = int(parts[0].strip()) if parts[0].strip().isdigit() else None
                m['away_goals'] = int(parts[1].strip()) if len(parts)>1 and parts[1].strip().isdigit() else None
            else:
                m['home_goals'] = None; m['away_goals'] = None
                
            matches.append(m)

        print(f"✅ [10maggio87] {season}: Trovate {len(matches)} partite (uniche).")
        return matches
    except Exception as e:
        print(f"❌ [10maggio87] Errore su {season}: {e}")
        return []

In [ ]:
# @title 5. Estrattore StadiaPostcards (Spettatori Serie A)
import requests
import pandas as pd
def scrape_stadiapostcards(season):
    parts = season.split('-')
    if len(parts) != 2: return []
    year_start = int(parts[0])
    if year_start < 1963: return [] 
    
    suffix = f'{str(year_start)[-2:]}-{parts[1]}'
    url = f'https://www.stadiapostcards.com/A{suffix}.htm'
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200: return []
        
        tables = pd.read_html(resp.text, flavor='lxml')
        sp_data = []
        for df in tables:
            if 'napoli' in df.astype(str).to_string().lower():
                for _, row in df.iterrows():
                    row_str = str(row.dropna().to_dict()).lower()
                    if 'napoli' in row_str:
                        sp_data.append({'source': 'stadiapostcards', 'raw': row.dropna().to_dict()})
        print(f"✅ [StadiaPostcards] {season}: {len(sp_data)} righe identificate.")
        return sp_data
    except:
        return []

In [ ]:
# @title 6. Riconciliazione e Inserimento in SQLite
import sqlite3
import json

def reconcile_and_save(season, m87_matches, sp_data):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    for m in m87_matches:
        date_clean = m.get('date', '1926-01-01').replace('-', '')
        home_short = m.get('squadra_casa', 'XXX')[:3].upper()
        away_short = m.get('squadra_trasferta', 'XXX')[:3].upper()
        match_id = f"{date_clean}-{home_short}-{away_short}"
        
        fonti_usate = ["10maggio87"]
        affluenza = None
        note = []
        
        cursor.execute("""
            INSERT OR REPLACE INTO matches 
            (match_id, date, season, competizione, squadra_casa, squadra_trasferta, home_goals, away_goals, affluenza, fonti, note)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            match_id,
            m.get('date'),
            m.get('season'),
            m.get('competizione'),
            m.get('squadra_casa'),
            m.get('squadra_trasferta'),
            m.get('home_goals'),
            m.get('away_goals'),
            affluenza,
            json.dumps(fonti_usate),
            json.dumps(note)
        ))
        
    conn.commit()
    conn.close()

In [ ]:
# @title 7. Esecuzione Master Loop
import time
stagioni_da_estrarre = ["1986-87", "1989-90", "2024-25"]

for season in stagioni_da_estrarre:
    print(f"\n⏳ Avvio estrazione stagione {season}...")
    m87_matches = scrape_10maggio87(season)
    time.sleep(2)
    sp_data = scrape_stadiapostcards(season)
    time.sleep(2)
    reconcile_and_save(season, m87_matches, sp_data)
    print(f"✅ {season} archiviata in SQLite.")

In [ ]:
# @title 8. Esportazione JSON per il Frontend
import sqlite3
import json
from google.colab import files

def export_matches_to_json():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM matches ORDER BY date DESC")
    rows = cursor.fetchall()
    matches = [dict(row) for row in rows]
    conn.close()
    
    for m in matches:
        m['fonti'] = json.loads(m['fonti']) if m.get('fonti') else []
        m['note'] = json.loads(m['note']) if m.get('note') else []
        m['scorers'] = json.loads(m['scorers']) if m.get('scorers') else []
        
    output_path = f"{BASE_DIR}/processed/matches.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(matches, f, ensure_ascii=False, indent=2)
        
    print(f"\n🎉 Esportate {len(matches)} partite nel file: {output_path}")
    files.download(output_path)

export_matches_to_json()